# get data

In [1]:
import pandas as pd
import requests

In [2]:
url = "https://api.openml.org/data/download/22120560/dataset"
output_file = "../data/dataset.arff"

response = requests.get(url)
response.raise_for_status()

with open(output_file, "wb") as file:
    file.write(response.content)

In [3]:
import arff
import pandas as pd

file_path = "../data/dataset.arff"

with open(file_path, "r", encoding="utf-8") as file:
    arff_data = arff.load(file)

df = pd.DataFrame(
    arff_data["data"],
    columns=[attribute[0] for attribute in arff_data["attributes"]]
)

print(df.head())
print(df.dtypes)


   Age     Sex  Job Housing Saving accounts Checking account  Credit amount  \
0   67    male    2     own             NaN           little           1169   
1   22  female    2     own          little         moderate           5951   
2   49    male    1     own          little              NaN           2096   
3   45    male    2    free          little           little           7882   
4   53    male    2    free          little           little           4870   

   Duration              Purpose  Risk  
0         6             radio/TV  good  
1        48             radio/TV   bad  
2        12            education  good  
3        42  furniture/equipment  good  
4        24                  car   bad  
Age                 int64
Sex                   str
Job                 int64
Housing               str
Saving accounts       str
Checking account      str
Credit amount       int64
Duration            int64
Purpose               str
Risk                  str
dtype: object


In [4]:
string_columns = df.select_dtypes(include=["string", "object"]).columns

df[string_columns] = df[string_columns].astype("object")

In [5]:
df.head()

,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose,Risk
0,67,male,2,own,NaN,little,1169,6,radio/TV,good
1,22,female,2,own,little,moderate,5951,48,radio/TV,bad
2,49,male,1,own,little,NaN,2096,12,education,good
3,45,male,2,free,little,little,7882,42,furniture/equipment,good
4,53,male,2,free,little,little,4870,24,car,bad


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Age               1000 non-null   int64 
 1   Sex               1000 non-null   object
 2   Job               1000 non-null   int64 
 3   Housing           1000 non-null   object
 4   Saving accounts   817 non-null    object
 5   Checking account  606 non-null    object
 6   Credit amount     1000 non-null   int64 
 7   Duration          1000 non-null   int64 
 8   Purpose           1000 non-null   object
 9   Risk              1000 non-null   object
dtypes: int64(4), object(6)
memory usage: 78.3+ KB


# target variable

In [7]:
df.groupby('Risk').size()

Risk
bad     300
good    700
dtype: int64

In [8]:
risk_dummies = pd.get_dummies(
    df["Risk"],
    dtype=int
)
risk_dummies.head()

,bad,good
0,0,1
1,1,0
2,0,1
3,0,1
4,1,0


In [9]:
df = pd.concat(
    [df.drop(columns="Risk"), risk_dummies],
    axis=1
)

In [10]:
df.head()

,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose,bad,good
0,67,male,2,own,NaN,little,1169,6,radio/TV,0,1
1,22,female,2,own,little,moderate,5951,48,radio/TV,1,0
2,49,male,1,own,little,NaN,2096,12,education,0,1
3,45,male,2,free,little,little,7882,42,furniture/equipment,0,1
4,53,male,2,free,little,little,4870,24,car,1,0


# store data

In [11]:
df.shape

(1000, 11)

In [12]:
df.to_pickle("../data/00_df_data.pkl")

In [13]:
import hashlib

with open("../data/00_df_data.pkl", "rb") as file:
    file_hash = hashlib.sha256(file.read()).hexdigest()

print(file_hash)

fa8ad6f425df8a1783ff3d51abbd0f9191a0b102e8c5a0128da6186c2a00becf


# data split into training and test

In [14]:
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["bad"]
)

In [15]:
train_df.to_pickle("../data/00_df_train.pkl")
test_df.to_pickle("../data/00_df_test.pkl")

In [16]:
train_df.groupby('bad').size()

bad
0    560
1    240
dtype: int64

In [17]:
test_df.groupby('bad').size()

bad
0    140
1     60
dtype: int64